**Installing required libs**

In [ ]:
!pip install ultralytics
!pip install gdown

**Verify NVIDIA GPU Availability**

In [ ]:
!nvidia-smi

**Downloading a dataset from GoogleDrive**

In [ ]:
!gdown https://drive.google.com/file/d/1tRvPHNL5lJPJGoBw0_PmMp_y_-t6-6af/view
!unzip -o -q 'data.zip' -d '/content/custom_data'

**Split images into train and validation folders**

In [ ]:
!curl -o /content/train_val_split.py https://raw.githubusercontent.com/mstya/yolo-candy-detector/refs/heads/main/src/train_val_split.py

In [ ]:
!python /content/train_val_split.py --targetpath="/content/split-data" --datapath="/content/custom_data/data" --train_pct=0.9

**Configure Training**

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

root = '/content/'

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': f'/{root}/custom_data/data',
      'train': f'/{root}/split-data/train/images',
      'val': f'/{root}/split-data/validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = f'/{root}/custom_data/data/classes.txt'
path_to_data_yaml = f'/{root}/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

**Training**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

**Test model**

In [ ]:
!yolo detect predict model=runs/detect/train-2/weights/best.pt source=/content/split-data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'runs/detect/predict-2/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

In [ ]:
!wget -O /content/yolo_detect.py https://raw.githubusercontent.com/mstya/yolo-candy-detector/refs/heads/main/src/yolo_detect.py

In [ ]:
!python /content/yolo_detect.py --model runs/detect/train-2/weights/best.pt --source /content/video.mp4 --resolution 1280x720